# Qwen视觉模型和动态FPS视频

Qwen-VL 族是2026年最具影响力的开源视频语言模型。每一代都提出了一个架构决定性赌注，在12个月内就被其他生态抄过去：通过M-RoPE实现的动态分辨率，带绝对时间对齐的动态FPS采样，ViT中的窗口注意力以及结构化的agent输出格式。到Qwen3-VL，架构基本稳定了：一个2D-RoPE-ViT编码器能处理自然长宽比输入，一个MLP投影到Qwen3语言基，以及强调OCR、grounding以及agent行为的训练步骤。

## 问题描述

QWen-VL的上线是对LLaVA-1.5和BLIP-2的相应，Qwen团队的目标在三个领域：分辨率、视频和结构化输出。

### 分辨率

LLaVA-1.5 在336x336分辨率上运行。对于图片够用，但是对一张中文发表或者稠密电子表格的截图不管用。QWen-VL的第一个改进是448x448分辨率以及grounded包围盒输出，让模型指着东西说话。

### 视频

Video-LLaMA 将每帧的解码结果叠加起来，然后喂给LLM。对于短片有用，但是对于几分钟的视频，时间轴也携带了信号。Qwen团队想要一个能够理解时间的编码器。

### 结构化输出

LLaVA输出的是自由格式语言。但是Agent需要JSON格式。Qwen-VL训练成输出包含包围盒坐标的显式JSON格式。

每一代Qwen-VL 都在三个轴中的一个方向做出改进。

## 基本概念

### Qwen-VL

第一代的贡献：
- 448x448分辨率
- Grounding   在图文对上训练，输出显式的坐标token。“猫位于（112，204），（280，344）”
- 支持中文和英文多语言

### Qwen2-VL ————M-RoPE和自然分辨率

第二代将固定分辨率以及Q-Fromer改成了一个支持原生动态分辨率的ViT编码器，主要改动：
- 原生动态分辨率。  ViT支持可以倍28（或者2倍空间融合14）整除的任何HxW分辨率图片，一张1120x672的图片产生960个视觉token。没有缩放、没有切片没有缩略图。
- M-RoPE（多模态RoPE）。 每一个token包含三维位置信息（t，h，w）而不是一维。对于图片，t为0。对于视频，t是帧索引。RePE以每轴频率旋转Q/K向量。没有位置编码表。
- MLP投影。 丢掉Q-Former，直接用2层MLP融合patch tokens。
- 动态FPS视频。  视频默认以1-2FPS进行采样，但是模型接收任意数量的帧。

### Qwen2.5-VL ————动态FPS + 绝对时间

2.5代在视频上有了重大改进。动态FPS不只是“需要时采样更多帧”。
- 绝对时间token。  不再以位置索引（帧0，1，2）当成时间戳。而是“0:04的时候，猫跳了”。模型看见`<time>0.04</time>`交错在视觉token间。
- 动态FPS。 对于慢镜头，以1FPS采样，对于动作镜头，以4FPS采样，用户和训练时可选，M-RoPE可以自适应。
- ViT中的窗口注意力。 空间注意力做窗口化，每隔几层做一次全局注意力。
- 显式的JSON输出格式。 
- MRoPE-v2 缩放。位置随最大输入尺寸缩放，所以一段10分钟的视频不会耗尽频率范围。

### Qwen3-VL

增量升级。更大的LLM骨干，扩充的训练数据，改进的OCR。
重点放在数据和训练改进上，而非架构。

### M-RoPE 数学

经典的RoPE用成对坐标，位置m旋转一个维度为d的query q：
```
q_rot[2i] = q[2i] * cos(m * theta_i) - q[2i+1] * sin(m * theta_i)
q_rot[2i + 1] = q[2i] * sin(m * theta_i) + q[2i + 1] * cos(m * theta_i)
theta_i = 10000^(-2i/d)
```

M-RoPE 把隐藏维度分成三个频带，比如d=96，分32维给时间，32维给高，32维给宽。每个频带按它自己的轴位置旋转。一个在（t=5,h=10,w=20）的patch，其三个频带分别施加旋转`R_t(5),R_h(10),R_w(20)`。

文本token 用`t=text_index, h =0, w = 0`或某种归一化选择，保持兼容。视频帧用t=frame_time, h=row, w=col，单图用t=0。

好处在于一套位置编码就能处理文本、图像和视频。不用分支代码或不同的位置表。

### 动态FPS采样逻辑

给定视频时长T和目标token预算B：
- 计算你能负担的最大FPS： `fps_max = B / (T * tokens_per_frame)`。
- 从目标FPS表中`{1, 2, 4, 8}`选出满足`fps <= fps_max`的目标帧率。
- 如果动作强烈（光流式启发或者用户显式请求），使用更高的fps。如果动作很慢，选低的。
- 根据选中的FPS均匀采样，在帧与帧之间插入`<time>t</time>`词元。

Qwen2.5-VL 隐式地训练这套逻辑；推理时用户通过fps参数控制，一段60秒动作序列以4FPS，每帧81 token = 19440 token，在32k的上下文中可控。

### 结构化Agent输出

Qwen2.5-VL的agent训练明确瞄准结构化工具调用：
```json
{
    "tool": "mouse_click",
    "coords": [1024, 512],
    "button": "left",
    "modifier": null
}
```
对比自由形式的"在(1024,512)处点击"，不需要正则匹配和歧义处理。

# 开始编码

教学积木：Qwen-VL 核心——**原生动态分辨率 ViT patch**、**三维 M-RoPE**、**动态 FPS + 绝对时间词元**、**结构化 agent JSON**。


## 1. 原生动态分辨率：任意 H×W（可被 patch 整除）→ 视觉 token + (t,h,w)


In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class TinyQwenVLConfig:
    """Qwen-VL 教学配置（尺寸偏小）。"""

    patch_size: int = 8
    """空间 patch；真 Qwen2-VL 常与 14/合并后的 28 相关。"""

    vit_dim: int = 48
    """视觉 token 维；也用作 head 总维示意。"""

    llm_dim: int = 64
    n_heads: int = 4
    rope_base: float = 10000.0
    time_token_id: int = 1
    """绝对时间占位 id（真系统是 ``<time>..</time>`` 文本词元）。"""
    vocab_size: int = 64


def assert_divisible(size: int, patch: int, name: str) -> None:
    """
    Args:
        size: 边长。
        patch: patch 大小。
        name: 报错用名称。
    """
    if size % patch != 0:
        raise ValueError(f"{name}={size} must be divisible by patch_size={patch}")


class NativeDynamicPatchEmbed(nn.Module):
    """
    原生动态分辨率 patch 嵌入：不强制缩放到固定方图、不做 AnyRes 切块。

    输入 ``(3, H, W)`` 且 H、W 可被 ``patch_size`` 整除即可。
    """

    def __init__(self, cfg: TinyQwenVLConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.proj = nn.Linear(3 * cfg.patch_size * cfg.patch_size, cfg.vit_dim)

    def forward(
        self,
        image: torch.Tensor,
        time_id: int = 0,
    ) -> tuple[torch.Tensor, torch.Tensor, tuple[int, int]]:
        """
        Args:
            image: ``(3, H, W)``，``H % P == 0`` 且 ``W % P == 0``。
            time_id: 帧时间索引（单图为 0；视频为帧号或量化时间步）。

        Returns:
            tokens: ``(Gh*Gw, vit_dim)``。
            pos_thw: ``(Gh*Gw, 3)``，每行为 ``(t, h, w)``。
            grid: ``(Gh, Gw)``。
        """
        if image.ndim != 3:
            raise ValueError(f"expect (3,H,W), got {tuple(image.shape)}")
        p = self.cfg.patch_size
        _, H, W = image.shape
        assert_divisible(H, p, "H")
        assert_divisible(W, p, "W")
        gh, gw = H // p, W // p
        x = image.reshape(3, gh, p, gw, p).permute(1, 3, 0, 2, 4).reshape(gh * gw, -1)
        tokens = self.proj(x)
        pos = []
        for r in range(gh):
            for c in range(gw):
                pos.append((time_id, r, c))
        pos_thw = torch.tensor(pos, device=image.device, dtype=torch.long)
        return tokens, pos_thw, (gh, gw)


print("NativeDynamicPatchEmbed ready")


## 2. M-RoPE：隐藏维拆成 t/h/w 三个频带分别旋转 Q/K


In [ ]:
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """
    Args:
        x: ``(..., rope_dim)``，最后一维为偶数。

    Returns:
        y: 与 ``x`` 同形。
    """
    x1 = x[..., ::2]
    x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)


def apply_mrope_bands(
    q: torch.Tensor,
    k: torch.Tensor,
    pos_thw: torch.Tensor,
    rope_dim: int,
    base: float = 10000.0,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    简化 M-RoPE：将 ``rope_dim`` 均分到三个轴 (t, h, w)，各自用对应坐标旋转。

    Args:
        q: ``(B, n_heads, L, head_dim)``。
        k: ``(B, n_heads, L, head_dim)``。
        pos_thw: ``(L, 3)``。
        rope_dim: 参与旋转的末维长度（偶数，``<= head_dim``，最好能被 6 整除以便三轴对称）。
        base: RoPE 基数。

    Returns:
        q_rot, k_rot: 与输入同形。
    """
    if rope_dim % 2:
        raise ValueError("rope_dim must be even")
    if rope_dim > q.size(-1):
        raise ValueError("rope_dim larger than head_dim")

    # 复数对数均分到三轴
    n_complex = rope_dim // 2
    base_split = n_complex // 3
    splits = [base_split, base_split, n_complex - 2 * base_split]

    def axis_angles(pos: torch.Tensor, n: int) -> torch.Tensor:
        if n == 0:
            return pos.new_zeros(pos.size(0), 0)
        idx = torch.arange(n, device=pos.device, dtype=torch.float32)
        inv = 1.0 / (base ** (idx / max(n, 1)))
        return pos.float().unsqueeze(1) * inv.unsqueeze(0)

    t, h, w = pos_thw[:, 0], pos_thw[:, 1], pos_thw[:, 2]
    ang = torch.cat(
        [axis_angles(t, splits[0]), axis_angles(h, splits[1]), axis_angles(w, splits[2])],
        dim=-1,
    )
    cos = torch.cos(ang).to(dtype=q.dtype)
    sin = torch.sin(ang).to(dtype=q.dtype)
    cos = torch.stack((cos, cos), dim=-1).flatten(-2)[None, None, :, :]
    sin = torch.stack((sin, sin), dim=-1).flatten(-2)[None, None, :, :]

    def rot(x: torch.Tensor) -> torch.Tensor:
        xr, xp = x[..., :rope_dim], x[..., rope_dim:]
        xr = xr * cos + rotate_half(xr) * sin
        return torch.cat([xr, xp], dim=-1)

    return rot(q), rot(k)


class MRoPEAttention(nn.Module):
    """带 M-RoPE 的单层自注意力（教学用）。"""

    def __init__(self, dim: int, n_heads: int, rope_base: float) -> None:
        super().__init__()
        if dim % n_heads:
            raise ValueError("dim must divide n_heads")
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.rope_dim = self.head_dim - (self.head_dim % 6)  # 尽量被 6 整除
        self.rope_base = rope_base
        self.qkv = nn.Linear(dim, dim * 3)
        self.out = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x: torch.Tensor, pos_thw: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: ``(B, L, D)``。
            pos_thw: ``(L, 3)``。

        Returns:
            y: ``(B, L, D)``。
        """
        B, L, D = x.shape
        qkv = self.qkv(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q, k = apply_mrope_bands(q, k, pos_thw, self.rope_dim, self.rope_base)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim**-0.5)
        attn = attn.softmax(dim=-1)
        h = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.norm(x + self.out(h))


print("M-RoPE ready")


## 3. 动态 FPS + 绝对时间词元交错；MLP 投影；结构化 Agent JSON


In [ ]:
def choose_dynamic_fps(
    duration_sec: float,
    tokens_per_frame: int,
    token_budget: int,
    candidates: tuple[int, ...] = (1, 2, 4, 8),
    prefer_high_motion: bool = False,
) -> int:
    """
    按笔记逻辑选 FPS：``fps_max = budget / (T * tokens_per_frame)``，再取候选中不超过上限者。

    Args:
        duration_sec: 视频时长（秒）。
        tokens_per_frame: 每帧视觉 token 数。
        token_budget: 视觉侧 token 预算。
        candidates: 可选 FPS 表。
        prefer_high_motion: True 时在可行集合中偏高 FPS。

    Returns:
        fps: 选中的帧率。
    """
    if duration_sec <= 0:
        raise ValueError("duration_sec must be positive")
    fps_max = token_budget / (duration_sec * tokens_per_frame)
    feasible = [f for f in candidates if f <= fps_max]
    if not feasible:
        return min(candidates)
    return max(feasible) if prefer_high_motion else min(feasible)


def sample_frame_indices(
    num_src_frames: int,
    duration_sec: float,
    fps: int,
) -> tuple[list[int], list[float]]:
    """
    按所选 FPS 在源帧上均匀采样，并给出绝对时间（秒）。

    Args:
        num_src_frames: 原始帧数。
        duration_sec: 视频时长。
        fps: 目标采样帧率。

    Returns:
        indices: 源帧下标列表。
        times_sec: 与各采样帧对齐的绝对时间（秒）。
    """
    n_out = max(1, int(duration_sec * fps))
    n_out = min(n_out, num_src_frames)
    if n_out == 1:
        return [0], [0.0]
    idxs = [
        int(round(i * (num_src_frames - 1) / (n_out - 1)))
        for i in range(n_out)
    ]
    times = [i * duration_sec / max(n_out - 1, 1) for i in range(n_out)]
    return idxs, times


def interleave_time_and_vision(
    vision_per_frame: list[torch.Tensor],
    times_sec: list[float],
    time_embed: nn.Embedding,
    time_token_id: int,
    time_resolution: float = 0.01,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    在帧与帧的视觉 token 之间插入绝对时间嵌入（模拟 ``<time>0.04</time>``）。

    Args:
        vision_per_frame: 长度 T，每个 ``(Nf, D)``。
        times_sec: 长度 T 的绝对时间（秒）。
        time_embed: 把量化时间 id 映到 ``D`` 维。
        time_token_id: 未直接使用的占位（保留与词表对齐的语义钩子）。
        time_resolution: 量化步长（秒），默认 0.01 → ``0.04`` 对应 id 相关编码。

    Returns:
        sequence: ``(T + sum Nf, D)``，布局 ``[time_0, vis_0..., time_1, vis_1..., ...]``。
        pos_thw: ``(L, 3)``；时间词元用 ``(t_quant, 0, 0)``，视觉用各帧 ``(t_idx, h, w)`` 已在外构造时可重算。
            这里简化：时间位置 ``(round(t/res), 0, 0)``，视觉沿用帧索引作 t。
    """
    if len(vision_per_frame) != len(times_sec):
        raise ValueError("vision_per_frame and times_sec length mismatch")
    chunks: list[torch.Tensor] = []
    pos_rows: list[tuple[int, int, int]] = []
    for t_idx, (vis, t_sec) in enumerate(zip(vision_per_frame, times_sec)):
        q = int(round(t_sec / time_resolution))
        # 用量化时间当 embedding 索引（clamp 到词表）
        tid = torch.tensor([min(q, time_embed.num_embeddings - 1)], device=vis.device)
        te = time_embed(tid)  # (1, D)
        chunks.append(te)
        pos_rows.append((q, 0, 0))
        chunks.append(vis)
        # 视觉：t 用帧序号；h/w 用线性下标拆不开时填 0 占位——调用方应传入已带网格的 pos
        for i in range(vis.size(0)):
            pos_rows.append((t_idx, i, 0))
    seq = torch.cat(chunks, dim=0)
    pos = torch.tensor(pos_rows, device=seq.device, dtype=torch.long)
    return seq, pos


class MLPProjector(nn.Module):
    """Qwen2-VL 式丢掉 Q-Former，两层 MLP 投到语言模型维。"""

    def __init__(self, vit_dim: int, llm_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(vit_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """
        Args:
            tokens: ``(N, vit_dim)``。

        Returns:
            out: ``(N, llm_dim)``。
        """
        return self.net(tokens)


def build_agent_click_json(
    x: int,
    y: int,
    button: str = "left",
    modifier: str | None = None,
) -> dict[str, Any]:
    """
    构造笔记中的结构化 agent 输出（非自由文本）。

    Args:
        x: 点击横坐标。
        y: 点击纵坐标。
        button: 鼠标键。
        modifier: 修饰键，可为 ``None``。

    Returns:
        payload: 可 ``json.dumps`` 的字典。
    """
    return {
        "tool": "mouse_click",
        "coords": [x, y],
        "button": button,
        "modifier": modifier,
    }


class TinyQwenVLFrontEnd(nn.Module):
    """把动态分辨率编码、M-RoPE 注意力与 MLP 投影串起来的教学前端。"""

    def __init__(self, cfg: TinyQwenVLConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.patch = NativeDynamicPatchEmbed(cfg)
        self.attn = MRoPEAttention(cfg.vit_dim, cfg.n_heads, cfg.rope_base)
        self.proj = MLPProjector(cfg.vit_dim, cfg.llm_dim)
        self.time_embed = nn.Embedding(4096, cfg.vit_dim)

    def encode_image(self, image: torch.Tensor) -> torch.Tensor:
        """
        Args:
            image: ``(3, H, W)``。

        Returns:
            llm_tokens: ``(N, llm_dim)``。
        """
        tok, pos, _ = self.patch(image, time_id=0)
        h = self.attn(tok.unsqueeze(0), pos).squeeze(0)
        return self.proj(h)

    def encode_video(
        self,
        frames: torch.Tensor,
        duration_sec: float,
        token_budget: int,
        prefer_high_motion: bool = False,
    ) -> tuple[torch.Tensor, int, list[float]]:
        """
        Args:
            frames: ``(T_src, 3, H, W)``，H/W 可被 patch 整除。
            duration_sec: 时长。
            token_budget: 视觉预算（不含时间词元时的近似）。
            prefer_high_motion: 是否偏向更高 FPS。

        Returns:
            llm_tokens: ``(L, llm_dim)``（含交错时间嵌入再投影）。
            fps: 选中的 FPS。
            times_sec: 采样绝对时间列表。
        """
        # 先编码第 0 帧估每帧 token 数
        t0, _, grid = self.patch(frames[0], time_id=0)
        tokens_per_frame = t0.size(0)
        fps = choose_dynamic_fps(
            duration_sec,
            tokens_per_frame,
            token_budget,
            prefer_high_motion=prefer_high_motion,
        )
        idxs, times = sample_frame_indices(frames.size(0), duration_sec, fps)
        vis_list: list[torch.Tensor] = []
        for t_idx, src_i in enumerate(idxs):
            tok, pos, _ = self.patch(frames[src_i], time_id=t_idx)
            tok = self.attn(tok.unsqueeze(0), pos).squeeze(0)
            vis_list.append(tok)
        seq, _ = interleave_time_and_vision(
            vis_list,
            times,
            self.time_embed,
            self.cfg.time_token_id,
        )
        return self.proj(seq), fps, times


print("FPS / projector / agent JSON ready")


## 4. 冒烟测试


In [ ]:
def smoke_test() -> None:
    """验证动态分辨率、M-RoPE、动态 FPS 与结构化 JSON。"""
    torch.manual_seed(0)
    cfg = TinyQwenVLConfig()
    fe = TinyQwenVLFrontEnd(cfg)

    print("=== native dynamic resolution ===")
    img = torch.randn(3, 32, 48)  # 4x6 patches
    tok = fe.encode_image(img)
    print(f"image tokens -> llm: {tuple(tok.shape)}")
    assert tok.size(1) == cfg.llm_dim

    print("\n=== M-RoPE relative geometry ===")
    # 直接检验旋转：改 w 应改变 Q（残差注意力可能把输出差冲得很小）
    B, H, L = 1, cfg.n_heads, 8
    head_dim = cfg.vit_dim // cfg.n_heads
    q = torch.randn(B, H, L, head_dim)
    k = torch.randn(B, H, L, head_dim)
    pos = torch.tensor([[0, i // 4, i % 4] for i in range(L)], dtype=torch.long)
    rope_dim = head_dim - (head_dim % 6)
    q1, k1 = apply_mrope_bands(q, k, pos, rope_dim, cfg.rope_base)
    pos2 = pos.clone()
    pos2[:, 2] += 3
    q2, k2 = apply_mrope_bands(q, k, pos2, rope_dim, cfg.rope_base)
    assert not torch.allclose(q1, q2)
    assert not torch.allclose(k1, k2)
    print("changing w rotates Q/K differently: OK")

    print("\n=== dynamic FPS ===")
    frames = torch.randn(30, 3, 24, 24)
    # tokens_per_frame = (24/8)^2 = 9; budget 40 -> fps_max = 40/(10*9)≈0.44 -> min candidate 1
    out, fps, times = fe.encode_video(frames, duration_sec=10.0, token_budget=40, prefer_high_motion=False)
    print(f"fps={fps}, times[:3]={times[:3]}, out={tuple(out.shape)}")
    out2, fps2, _ = fe.encode_video(frames, duration_sec=5.0, token_budget=200, prefer_high_motion=True)
    print(f"short+high budget+motion -> fps={fps2}, out={tuple(out2.shape)}")
    assert fps2 >= fps

    payload = build_agent_click_json(1024, 512)
    print("\n=== agent JSON ===")
    print(json.dumps(payload, ensure_ascii=False))
    assert payload["tool"] == "mouse_click" and payload["coords"] == [1024, 512]
    print("SMOKE TEST OK")


smoke_test()
